# Graded Challenge 3 – Set 2
**Property Market Analysis: Rumah Dijual di Bekasi**

**Nama:** Nawafi Suren  
**Batch:** HCK-47




## 1. Import library


In [8]:
import numpy as np
import pandas as pd

from selenium import webdriver
from bs4 import BeautifulSoup

import time



## 2. Data scraping


In [ ]:

# 1. Import packages
from selenium import webdriver
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import time
from urllib.parse import urljoin


# 2. Ambil link dari halaman pencarian
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.set_page_load_timeout(30)

url = "https://www.rumah123.com/jual/cari/?location=bekasi"

try:
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

finally:
    driver.quit()


# 3. Kumpulkan URL detail properti
links = []

for a in soup.select('a[href*="/properti/"]'):

    link = urljoin(url, a["href"])

    if link not in links:
        links.append(link)

# Batasi data untuk percobaan
links = links[:100]

print("Jumlah link:", len(links))


# 4. Siapkan requests
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

data = []


# 5. Ambil informasi setiap properti
for link in links:

    try:
        response = session.get(link, timeout=15)
        response.raise_for_status()

        detail = BeautifulSoup(response.text, "html.parser")

        text = detail.get_text(" ", strip=True)

        # Ambil judul
        title = detail.find("h1")

        if title:
            title = title.get_text(strip=True)
        else:
            title = None

        # Ambil lokasi
        location = None

        if detail.find("h1"):
            next_text = detail.find("h1").find_next(string=True)

            if next_text:
                location = next_text.strip()

        # Fungsi mengambil angka berdasarkan label
        def ambil(label):
            hasil = re.search(
                label + r"\s*([\d.,]+)",
                text,
                re.IGNORECASE
            )

            return hasil.group(1) if hasil else None

        # Ambil harga
        price = re.search(
            r"Rp\s*[\d.,]+\s*(?:Triliun|Miliar|Juta)",
            text,
            re.IGNORECASE
        )

        # Simpan data
        data.append({
            "Judul": title,
            "Lokasi": location,
            "Luas Tanah": ambil("Luas Tanah"),
            "Luas Bangunan": ambil("Luas Bangunan"),
            "Carport": ambil("Carport"),
            "Jumlah Kamar Tidur": ambil("Kamar Tidur"),
            "Jumlah Kamar Mandi": ambil("Kamar Mandi"),
            "Harga Rumah": price.group(0) if price else None,
            "URL": link
        })

        print("Berhasil:", title)

        time.sleep(1)

    except Exception as e:
        print("Gagal:", link, e)


# 6. Buat DataFrame
df = pd.DataFrame(data)


# 7. Simpan hasil scraping
df.to_csv(
    "P0G3_data_nawafi_suren.csv",
    index=False,
    encoding="utf-8-sig"
)


# 8. Tampilkan hasil
display(df)

print("Total data:", len(df))

session.close()


Jumlah link: 24
Berhasil: Cluster Leyora Kota Wisata Ecovia
Berhasil: Nuone At Arcova Grand Wisata
Berhasil: Altara Grand Wisata
Berhasil: Rumah Deket dengan Summarecon Bekasi KPR DP 0%
Berhasil: Cukup 5 Juta Langsung Bawa Koper Pasilitas Lengkap Free Semi Furnish Free Mas 5 Gram Kamar Tidur 2 Kamar Mandi 1 di Bantu Lolos KPR di Mutiara Gading City
Berhasil: Rumah Mewah Siap Huni Cluster Magenta Summarecon Bekasi 6X11 Full Renov + Rooftop
Berhasil: Di Jual Rumah Burgundy Residence Summareconbekasi
Berhasil: Livia at Ladoria Grand Wisata
Berhasil: Di Jual Rumah Lotus Residencesummareconbekasi
Berhasil: Siap Huni Rumah Tipe Myzora 45 2 Lantai di Cibitung, Murah Under 1M
Berhasil: Rumah Baru View Danau Premium di Cluster Cresta Harapan Indah Bekasi - Mulai 1 M-An!!
Berhasil: Harga Miring!! Rumah Cluster Hook 114M2 di Cluster Samata ( Samping Cluster Cresta)
Berhasil: Hot Selling! Rumah Hook, Luas 192M2, Row Jalan 10M&20M, di Sekitaran Harapan Indah!!
Berhasil: Rumah Dalam Kondisi Baru Dij

,Judul,Lokasi,Luas Tanah,Luas Bangunan,Carport,Jumlah Kamar Tidur,Jumlah Kamar Mandi,Harga Rumah,URL
0,Cluster Leyora Kota Wisata Ecovia,Cluster Leyora Kota Wisata Ecovia,None,None,None,None,None,"Rp 4,93 Miliar",https://www.rumah123.com/perumahan-baru/proper...
1,Nuone At Arcova Grand Wisata,Nuone At Arcova Grand Wisata,None,None,None,None,None,Rp 980 Juta,https://www.rumah123.com/perumahan-baru/proper...
2,Altara Grand Wisata,Altara Grand Wisata,None,None,None,None,None,"Rp 1,5 Miliar",https://www.rumah123.com/perumahan-baru/proper...
3,Rumah Deket dengan Summarecon Bekasi KPR DP 0%,Rumah Deket dengan Summarecon Bekasi KPR DP 0%,60,36,1,2,1,Rp 435 Juta,https://www.rumah123.com/properti/bekasi-summa...
4,Cukup 5 Juta Langsung Bawa Koper Pasilitas Len...,Cukup 5 Juta Langsung Bawa Koper Pasilitas Len...,60,32,1,2,1,Rp 402 Juta,https://www.rumah123.com/properti/bekasi-harap...
5,Rumah Mewah Siap Huni Cluster Magenta Summarec...,Rumah Mewah Siap Huni Cluster Magenta Summarec...,66,95,1,4,2,"Rp 1,75 Miliar",https://www.rumah123.com/properti/bekasi-bekas...
6,Di Jual Rumah Burgundy Residence Summareconbekasi,Di Jual Rumah Burgundy Residence Summareconbekasi,55,52,1,2,2,"Rp 1,25 Miliar",https://www.rumah123.com/properti/bekasi-summa...
7,Livia at Ladoria Grand Wisata,Livia at Ladoria Grand Wisata,None,None,None,None,None,"Rp 2,1 Miliar",https://www.rumah123.com/perumahan-baru/proper...
8,Di Jual Rumah Lotus Residencesummareconbekasi,Di Jual Rumah Lotus Residencesummareconbekasi,162,200,2,4,3,"Rp 3,85 Miliar",https://www.rumah123.com/properti/bekasi-summa...
9,Siap Huni Rumah Tipe Myzora 45 2 Lantai di Cib...,Siap Huni Rumah Tipe Myzora 45 2 Lantai di Cib...,72,45,1,3,1,Rp 715 Juta,https://www.rumah123.com/properti/bekasi-cikar...


Total data: 24


In [3]:

# Informasi DataFrame
df.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Judul               20 non-null     object
 1   Lokasi              20 non-null     object
 2   Luas Tanah          16 non-null     object
 3   Luas Bangunan       16 non-null     object
 4   Carport             13 non-null     object
 5   Jumlah Kamar Tidur  16 non-null     object
 6   Jumlah Kamar Mandi  16 non-null     object
 7   Harga Rumah         20 non-null     object
 8   URL                 20 non-null     object
dtypes: object(9)
memory usage: 1.5+ KB


In [4]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Jumlah data:", len(df))

display(df.head())

Shape: (20, 9)
Columns: ['Judul', 'Lokasi', 'Luas Tanah', 'Luas Bangunan', 'Carport', 'Jumlah Kamar Tidur', 'Jumlah Kamar Mandi', 'Harga Rumah', 'URL']
Jumlah data: 20


,Judul,Lokasi,Luas Tanah,Luas Bangunan,Carport,Jumlah Kamar Tidur,Jumlah Kamar Mandi,Harga Rumah,URL
0,Cluster Leyora Kota Wisata Ecovia,Cluster Leyora Kota Wisata Ecovia,None,None,None,None,None,"Rp 4,93 Miliar",https://www.rumah123.com/perumahan-baru/proper...
1,Nuone At Arcova Grand Wisata,Nuone At Arcova Grand Wisata,None,None,None,None,None,Rp 980 Juta,https://www.rumah123.com/perumahan-baru/proper...
2,Altara Grand Wisata,Altara Grand Wisata,None,None,None,None,None,"Rp 1,5 Miliar",https://www.rumah123.com/perumahan-baru/proper...
3,Cukup 5 Juta Langsung Bawa Koper Pasilitas Len...,Cukup 5 Juta Langsung Bawa Koper Pasilitas Len...,60,32,1,2,1,Rp 402 Juta,https://www.rumah123.com/properti/bekasi-harap...
4,Rumah Mewah Siap Huni Cluster Magenta Summarec...,Rumah Mewah Siap Huni Cluster Magenta Summarec...,66,95,1,4,2,"Rp 1,75 Miliar",https://www.rumah123.com/properti/bekasi-bekas...


In [7]:
df.info()
display(df.describe(include="all").T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Judul               20 non-null     object
 1   Lokasi              20 non-null     object
 2   Luas Tanah          16 non-null     object
 3   Luas Bangunan       16 non-null     object
 4   Carport             13 non-null     object
 5   Jumlah Kamar Tidur  16 non-null     object
 6   Jumlah Kamar Mandi  16 non-null     object
 7   Harga Rumah         20 non-null     object
 8   URL                 20 non-null     object
dtypes: object(9)
memory usage: 1.5+ KB


,count,unique,top,freq
Judul,20,20,Cluster Leyora Kota Wisata Ecovia,1
Lokasi,20,20,Cluster Leyora Kota Wisata Ecovia,1
Luas Tanah,16,10,60,4
Luas Bangunan,16,12,200,2
Carport,13,3,1,8
Jumlah Kamar Tidur,16,4,2,5
Jumlah Kamar Mandi,16,3,1,6
Harga Rumah,20,16,"Rp 1,5 Miliar",4
URL,20,20,https://www.rumah123.com/perumahan-baru/proper...,1
